Phase 3 — Accuracy Evaluation

Goal:
Evaluate Top-1 accuracy retention for:

1. Baseline FP32
2. Structural Pruning (P)
3. Structural Pruning → Quantization (P → Q)
4. Structural Quantization → Pruning (Q → P, repaired)

using an ImageNet-compatible validation dataset.

In [1]:
import torch
import torchvision
import torchvision.transforms as transforms

from torchvision.models import mobilenet_v2

In [2]:
print(torch.__version__)
print(torchvision.__version__)

2.11.0+cu126
0.26.0+cu126


In [3]:
from torchvision.datasets import ImageFolder

print("Ready")

Ready


In [4]:
import os

os.makedirs("../data", exist_ok=True)

print("Data folder ready.")

Data folder ready.


In [5]:
from torchvision.datasets import ImageFolder
from torchvision import transforms
from torchvision.datasets.utils import download_and_extract_archive
import os

In [6]:
DATA_DIR = "../data"

url = "https://s3.amazonaws.com/fast-ai-imageclas/imagenette2-160.tgz"

download_and_extract_archive(
    url=url,
    download_root=DATA_DIR,
    remove_finished=False
)

print("Download complete.")

Download complete.


In [7]:
import os
print(os.listdir("../data"))

['imagenette2-160', 'imagenette2-160.tgz']


In [8]:
transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [9]:
val_dataset = ImageFolder(
    "../data/imagenette2-160/val",
    transform=transform
)

print("Validation images:", len(val_dataset))

Validation images: 3925


In [10]:
from torch.utils.data import DataLoader

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=0
)

print("Validation loader ready.")

Validation loader ready.


In [11]:
print(val_dataset.classes)

['n01440764', 'n02102040', 'n02979186', 'n03000684', 'n03028079', 'n03394916', 'n03417042', 'n03425413', 'n03445777', 'n03888257']


In [12]:
baseline_model = mobilenet_v2(weights="DEFAULT")
baseline_model.eval()

print("Baseline model loaded.")

Baseline model loaded.


In [13]:
# =====================================================
# Map Imagenette's 10 classes to their ImageNet-1000 indices
# =====================================================

# Imagenette wnids -> corresponding index in the standard 1000-class
# ImageNet output that MobileNetV2 was trained on.
imagenette_wnid_to_imagenet_idx = {
    "n01440764": 0,    # tench
    "n02102040": 217,  # English springer
    "n02979186": 482,  # cassette player
    "n03000684": 491,  # chain saw
    "n03028079": 497,  # church
    "n03394916": 566,  # French horn
    "n03417042": 569,  # garbage truck
    "n03425413": 571,  # gas pump
    "n03445777": 574,  # golf ball
    "n03888257": 701,  # parachute
}

# val_dataset.classes is already sorted alphabetically by wnid, and
# ImageFolder assigns labels 0-9 in that same sorted order — so
# imagenet_indices[i] gives the correct 1000-class index for local label i.
imagenet_indices = [
    imagenette_wnid_to_imagenet_idx[wnid] for wnid in val_dataset.classes
]

print("Local label -> ImageNet index mapping:")
for local_label, imagenet_idx in enumerate(imagenet_indices):
    print(f"  {local_label} ({val_dataset.classes[local_label]}) -> {imagenet_idx}")

Local label -> ImageNet index mapping:
  0 (n01440764) -> 0
  1 (n02102040) -> 217
  2 (n02979186) -> 482
  3 (n03000684) -> 491
  4 (n03028079) -> 497
  5 (n03394916) -> 566
  6 (n03417042) -> 569
  7 (n03425413) -> 571
  8 (n03445777) -> 574
  9 (n03888257) -> 701


In [14]:
def evaluate_accuracy(model, dataloader, imagenet_indices, device="cpu"):
    model.eval()
    model.to(device)

    imagenet_indices_tensor = torch.tensor(imagenet_indices, device=device)

    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in dataloader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            # Only look at the 10 logits that correspond to Imagenette
            # classes, instead of comparing against all 1000.
            relevant_logits = outputs[:, imagenet_indices_tensor]

            # argmax over those 10 gives back a 0-9 local label directly.
            predicted = torch.argmax(relevant_logits, dim=1)

            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    return 100 * correct / total

In [15]:
baseline_accuracy = evaluate_accuracy(
    baseline_model,
    val_loader,
    imagenet_indices,
    "cpu"
)

print(f"Baseline Accuracy: {baseline_accuracy:.2f}%")

Baseline Accuracy: 97.35%


In [16]:
import torch.nn as nn
import torch_pruning as tp

Structural Pruning (P) only

In [18]:
p_model = mobilenet_v2(weights="DEFAULT")
p_model.eval()

example_inputs = torch.randn(1, 3, 224, 224)
importance = tp.importance.MagnitudeImportance(p=2)
ignored_layers = [p_model.classifier]

pruner = tp.pruner.MagnitudePruner(
    p_model,
    example_inputs,
    importance=importance,
    pruning_ratio=0.2,
    ignored_layers=ignored_layers
)
pruner.step()

p_accuracy = evaluate_accuracy(p_model, val_loader, imagenet_indices, "cpu")
print(f"Structural Pruning (P) Accuracy: {p_accuracy:.2f}%")

Structural Pruning (P) Accuracy: 8.38%


P → Q

In [20]:
pq_model = mobilenet_v2(weights="DEFAULT")
pq_model.eval()

pruner_pq = tp.pruner.MagnitudePruner(
    pq_model,
    example_inputs,
    importance=tp.importance.MagnitudeImportance(p=2),
    pruning_ratio=0.2,
    ignored_layers=[pq_model.classifier]
)
pruner_pq.step()

pq_model = torch.quantization.quantize_dynamic(
    pq_model,
    {torch.nn.Linear},
    dtype=torch.qint8
)

pq_accuracy = evaluate_accuracy(pq_model, val_loader, imagenet_indices, "cpu")
print(f"P → Q Accuracy: {pq_accuracy:.2f}%")

C:\Users\user\AppData\Local\Temp\ipykernel_14608\2725024549.py:13: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  pq_model = torch.quantization.quantize_dynamic(


P → Q Accuracy: 8.18%


Q → P (repaired)

In [22]:
pristine = mobilenet_v2(weights="DEFAULT")
pristine.eval()
original_classifier_weight = pristine.classifier[1].weight.detach().clone()
original_classifier_bias = pristine.classifier[1].bias.detach().clone()

qp_model = torch.quantization.quantize_dynamic(
    mobilenet_v2(weights="DEFAULT").eval(),
    {nn.Linear},
    dtype=torch.qint8
)

pruner_qp = tp.pruner.MagnitudePruner(
    qp_model,
    example_inputs,
    importance=tp.importance.MagnitudeImportance(p=2),
    pruning_ratio=0.2,
    ignored_layers=[qp_model.classifier]
)
pruner_qp.step()

history = pruner_qp.pruning_history()
removed_idxs = None
for layer_name, is_out_channel, idxs in history:
    if layer_name == "features.18.0":
        removed_idxs = idxs
        break

all_channels = set(range(1280))
kept_idxs = sorted(list(all_channels - set(removed_idxs)))
kept_idxs_tensor = torch.tensor(kept_idxs, dtype=torch.long)

sliced_weight = original_classifier_weight[:, kept_idxs_tensor]
sliced_bias = original_classifier_bias.clone()

repaired_float_linear = nn.Linear(in_features=len(kept_idxs), out_features=1000)
with torch.no_grad():
    repaired_float_linear.weight.copy_(sliced_weight)
    repaired_float_linear.bias.copy_(sliced_bias)

repaired_float_linear.qconfig = torch.quantization.default_dynamic_qconfig
repaired_quantized_linear = torch.ao.nn.quantized.dynamic.Linear.from_float(repaired_float_linear)

qp_model.classifier = nn.Sequential(
    nn.Dropout(0.2),
    repaired_quantized_linear
)

qp_accuracy = evaluate_accuracy(qp_model, val_loader, imagenet_indices, "cpu")
print(f"Q → P (Repaired) Accuracy: {qp_accuracy:.2f}%")

C:\Users\user\AppData\Local\Temp\ipykernel_14608\2546061919.py:6: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  qp_model = torch.quantization.quantize_dynamic(


Q → P (Repaired) Accuracy: 8.08%


Summary Table

In [24]:
import pandas as pd

summary = pd.DataFrame({
    "Experiment": ["Baseline FP32", "Structural Pruning (P)", "P → Q", "Q → P (Repaired)"],
    "Accuracy (%)": [baseline_accuracy, p_accuracy, pq_accuracy, qp_accuracy]
})
print(summary)

               Experiment  Accuracy (%)
0           Baseline FP32     97.350318
1  Structural Pruning (P)      8.382166
2                   P → Q      8.178344
3        Q → P (Repaired)      8.076433
